# Hybrid Semantic Parent-Child Retrieval -> GPT-4o-mini Answer

Notebook này là bản so sánh với `vi_to_en_retrieval_gpt4o_mini_pipeline.ipynb`.

Phần API, translate query, answer prompt và output loop được giữ theo notebook cũ. Phần thay đổi chính là retrieval:

1. Dùng FAISS/BM25/Hybrid trên child chunks của index `semantic_parent_child`.
2. Sau khi retrieve child, notebook đọc thêm `semantic_children.jsonl` và `semantic_parents.jsonl`.
3. Context đưa vào LLM gồm child evidence và parent context summary.
4. Parent không được embedding và không nằm trong FAISS.


In [46]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any

from openai import OpenAI


# =========================
# 1. Cấu hình pipeline
# =========================

def find_project_root(start: Path) -> Path:
    """Tìm thư mục gốc project dựa trên cấu trúc backend/data quen thuộc."""

    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "backend" / "rag").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Không tìm thấy project root. Hãy chạy notebook trong repo travel-agent.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Embedding baseline dùng PyTorch, không dùng TensorFlow.
# Thiết lập trước khi import sentence-transformers để tránh lỗi Keras 3.
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

# File query đầu vào: bộ 500 query người dùng hay hỏi.
INPUT_QUERY_PATH = PROJECT_ROOT / "data" / "evaluation" / "traveler_need_queries_500_en.jsonl"

# File output chứa câu hỏi, query đã dịch, answer và source retrieval.
OUTPUT_PATH = PROJECT_ROOT / "report" / "evaluate" / "hybrid_chunk_report" / "semantic_parent_child_hybrid_retrieval_gpt4o_mini_answers.json"

# Giới hạn số query để chạy thử. Đổi thành 100/500 khi muốn chạy nhiều hơn.
LIMIT = int(os.getenv("RAG_PIPELINE_LIMIT", "100"))

# Chọn retriever: dense, bm25 hoặc hybrid.
RETRIEVER_NAME = os.getenv("RAG_RETRIEVER", "hybrid").strip().lower()

# Số chunk đưa vào prompt trả lời.
TOP_K = int(os.getenv("RAG_TOP_K", "5"))
CANDIDATE_K = int(os.getenv("RAG_CANDIDATE_K", "5"))

# Giới hạn độ dài mỗi chunk khi đưa vào prompt để tránh prompt quá dài.
MAX_CHARS_PER_CHUNK = int(os.getenv("RAG_MAX_CHARS_PER_CHUNK", "1800"))

# Model chat mặc định theo yêu cầu.
CHAT_MODEL = os.getenv("CHAT_MODEL", "openai/gpt-4o-mini")

# Delay nhẹ để hạn chế rate limit.
REQUEST_DELAY_SECONDS = float(os.getenv("RAG_REQUEST_DELAY_SECONDS", "0.8"))

# Hybrid semantic parent-child retrieval files.
SEMANTIC_INDEX_DIR = PROJECT_ROOT / "data" / "indexes" / "paraphrase-multilingual-MiniLM-L12-v2_semantic_parent_child"
PARENTS_PATH = PROJECT_ROOT / "data" / "chunks" / "semantic_parents.jsonl"
CHILDREN_PATH = PROJECT_ROOT / "data" / "chunks" / "semantic_children.jsonl"
MAX_PARENT_SUMMARY_CHARS = int(os.getenv("RAG_MAX_PARENT_SUMMARY_CHARS", "700"))

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_QUERY_PATH)
print("Output:", OUTPUT_PATH)
print("LIMIT:", LIMIT, "| RETRIEVER:", RETRIEVER_NAME, "| TOP_K:", TOP_K, "| MODEL:", CHAT_MODEL)

Project root: D:\LLM\chatotAgentTravelling\code\travel-agent
Input: D:\LLM\chatotAgentTravelling\code\travel-agent\data\evaluation\traveler_need_queries_500_en.jsonl
Output: D:\LLM\chatotAgentTravelling\code\travel-agent\report\evaluate\hybrid_chunk_report\semantic_parent_child_hybrid_retrieval_gpt4o_mini_answers.json
LIMIT: 100 | RETRIEVER: hybrid | TOP_K: 5 | MODEL: openai/gpt-4o-mini


In [ ]:
# =========================
# 2. Khởi tạo API client
# =========================

def load_env_file(env_path: Path) -> None:
    """Đọc file .env đơn giản mà không cần thêm dependency python-dotenv."""

    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def load_api_config_from_test_model_notebook(notebook_path: Path) -> None:
    """Fallback: đọc cấu hình API từ backend/test_model.ipynb nếu notebook đó đã có token chạy được."""

    if not notebook_path.exists():
        return
    try:
        notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    except Exception:
        return

    code_text = "\n".join(
        "".join(cell.get("source", []))
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    )

    # Lấy token dạng: os.environ["GITHUB_TOKEN"] = "..."
    token_match = re.search(r"os\.environ\[[\"']GITHUB_TOKEN[\"']\]\s*=\s*[\"']([^\"']+)[\"']", code_text)
    if token_match:
        os.environ.setdefault("GITHUB_TOKEN", token_match.group(1))

    # Lấy base_url nếu có khai báo trong OpenAI(...).
    base_url_match = re.search(r"base_url\s*=\s*[\"']([^\"']+)[\"']", code_text)
    if base_url_match:
        os.environ.setdefault("GITHUB_MODELS_BASE_URL", base_url_match.group(1))

    # Lấy model mặc định từ test_model.ipynb nếu người dùng chưa override CHAT_MODEL.
    model_match = re.search(r"model\s*=\s*[\"']([^\"']+)[\"']", code_text)
    if model_match:
        os.environ.setdefault("CHAT_MODEL", model_match.group(1))


load_env_file(PROJECT_ROOT / ".env")
load_api_config_from_test_model_notebook(PROJECT_ROOT / "backend" / "test_model.ipynb")

# Nếu notebook chưa nhận biến môi trường, bạn có thể paste key vào đây để chạy ngay.
# Không commit/share notebook sau khi đã paste key thật.
# API_PROVIDER: "github", "openrouter" hoặc "openai".
API_PROVIDER = os.getenv("API_PROVIDER", "github").strip().lower()
API_KEYS_TEXT = """"""  # Ví dụ: """YOUR_GITHUB_TOKEN""" hoặc nhiều OpenRouter key, mỗi key một dòng.
API_BASE_URL = os.getenv("API_BASE_URL", "").strip()


def split_keys(value: str | None) -> list[str]:
    """Tách danh sách API key phân cách bằng dấu phẩy hoặc xuống dòng."""

    if not value:
        return []
    return [item.strip() for item in re.split(r"[,\n]+", value) if item.strip()]


def infer_provider_from_key(api_key: str, fallback_provider: str) -> str:
    """Tự nhận diện provider phổ biến để tránh paste key OpenRouter nhưng gửi sang GitHub."""

    if api_key.startswith("sk-or-v1-"):
        return "openrouter"
    if api_key.startswith("ghp_") or api_key.startswith("github_pat_"):
        return "github"
    return fallback_provider


def resolve_api_config() -> tuple[str, list[str]]:
    """Lấy API config từ key paste trong notebook, .env hoặc biến môi trường."""

    inline_keys = split_keys(API_KEYS_TEXT)
    if inline_keys:
        provider = infer_provider_from_key(inline_keys[0], API_PROVIDER)
        if provider == "openrouter":
            return API_BASE_URL or "https://openrouter.ai/api/v1", inline_keys
        if provider == "openai":
            return API_BASE_URL or "https://api.openai.com/v1", inline_keys
        return API_BASE_URL or "https://models.github.ai/inference", inline_keys

    openrouter_keys = split_keys(os.getenv("OPENROUTER_API_KEYS"))
    if openrouter_keys:
        return os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"), openrouter_keys

    github_token = os.getenv("GITHUB_TOKEN")
    if github_token:
        return os.getenv("GITHUB_MODELS_BASE_URL", "https://models.github.ai/inference"), [github_token]

    openai_keys = split_keys(os.getenv("OPENAI_API_KEYS")) or split_keys(os.getenv("OPENAI_API_KEY"))
    if openai_keys:
        return os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"), openai_keys

    raise RuntimeError(
        "Chưa có API key. Cách nhanh nhất: mở cell này, paste key vào API_KEYS_TEXT, "
        "chọn API_PROVIDER='github'/'openrouter'/'openai', rồi chạy lại từ cell 2."
    )


class RotatingChatClient:
    """Client OpenAI-compatible có hỗ trợ retry và đổi key khi gặp rate limit/quota tạm thời."""

    def __init__(self, base_url, api_keys, model):
        self.base_url = base_url.rstrip("/")
        self.api_keys = api_keys
        self.model = model
        self.key_index = 0

    def _client(self):
        return OpenAI(base_url=self.base_url, api_key=self.api_keys[self.key_index])

    def chat(self, messages, temperature=0.2, max_tokens=900):
        last_error = None
        retry_terms = ["too many requests", "rate", "quota", "limit", "429", "insufficient"]
        max_attempts = max(1, len(self.api_keys) * 3)
        for attempt in range(max_attempts):
            try:
                response = self._client().chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return response.choices[0].message.content or ""
            except Exception as exc:
                last_error = exc
                message = str(exc).lower()
                can_retry = any(term in message for term in retry_terms)
                if can_retry and attempt < max_attempts - 1:
                    if len(self.api_keys) > 1:
                        self.key_index = (self.key_index + 1) % len(self.api_keys)
                    wait_seconds = min(20, 2 * (attempt + 1))
                    print(f"API đang rate limit/quota tạm thời. Đợi {wait_seconds}s rồi thử lại...")
                    time.sleep(wait_seconds)
                    continue
                raise
        raise RuntimeError(f"Không gọi được chat model: {last_error}")


BASE_URL, API_KEYS = resolve_api_config()
chat_client = RotatingChatClient(BASE_URL, API_KEYS, CHAT_MODEL)
print("API endpoint:", BASE_URL)
print("Số API key đã nạp:", len(API_KEYS))

API endpoint: https://models.github.ai/inference
Số API key đã nạp: 1


In [39]:
# =========================
# 3. Khởi tạo retriever
# =========================

import importlib.util
import subprocess

# Ép Transformers dùng PyTorch-only. Dùng gán trực tiếp để ghi đè env cũ nếu có.
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"


def clear_transformers_import_cache():
    """Dọn module cache nếu kernel từng import lỗi TensorFlow/Keras trước đó."""

    prefixes = ("sentence_transformers", "transformers")
    for module_name in list(sys.modules):
        if module_name.startswith(prefixes):
            sys.modules.pop(module_name, None)


def ensure_package_installed(package_name, pip_name=None, version=None):
    """Đảm bảo package đã cài trong kernel hiện tại nhưng không import package đó quá sớm."""

    if importlib.util.find_spec(package_name) is not None:
        return

    install_name = pip_name or package_name
    if version:
        install_name = f"{install_name}=={version}"
    print(f"Thiếu package {package_name}. Đang cài {install_name} bằng kernel hiện tại...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", install_name])
    if importlib.util.find_spec(package_name) is None:
        raise ModuleNotFoundError(f"Đã cài {install_name} nhưng vẫn chưa import được {package_name}. Hãy restart kernel rồi chạy lại.")


clear_transformers_import_cache()

# FAISS là thư viện bắt buộc để đọc index.faiss của baseline retriever.
ensure_package_installed("faiss", pip_name="faiss-cpu", version="1.8.0.post1")

# Sentence Transformers là thư viện bắt buộc để embed query trước khi search FAISS.
ensure_package_installed("sentence_transformers", pip_name="sentence-transformers", version="3.2.1")

from backend.rag.retrieval.baseline_retrievers import RetrieverConfig, build_retrievers


retriever_config = RetrieverConfig(
    registry_path=PROJECT_ROOT / "configs" / "embedding_models.json",
    model_id="paraphrase-multilingual-MiniLM-L12-v2",
    index_dir=SEMANTIC_INDEX_DIR,
    device=os.getenv("RAG_EMBEDDING_DEVICE", "cpu"),
)

dense_retriever, bm25_retriever, hybrid_retriever = build_retrievers(retriever_config)
retrievers = {
    "dense": dense_retriever,
    "bm25": bm25_retriever,
    "hybrid": hybrid_retriever,
}
if RETRIEVER_NAME not in retrievers:
    raise ValueError(f"RETRIEVER_NAME phải là một trong {list(retrievers)}, hiện tại: {RETRIEVER_NAME}")

retriever = retrievers[RETRIEVER_NAME]
print("Đã khởi tạo retriever:", RETRIEVER_NAME)

Đã khởi tạo retriever: hybrid


In [40]:
# =========================
# 4. Hàm xử lý query, retrieval và answer
# =========================

def extract_json_object(text: str) -> dict[str, Any]:
    """Trích JSON object từ output LLM, kể cả khi model bọc thêm markdown."""

    clean_text = text.strip()
    if clean_text.startswith("```"):
        clean_text = re.sub(r"^```(?:json)?", "", clean_text).strip()
        clean_text = re.sub(r"```$", "", clean_text).strip()
    try:
        return json.loads(clean_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", clean_text, re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def load_queries(path: Path, limit: int) -> list[dict[str, Any]]:
    """Đọc query test set dạng JSONL."""

    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            rows.append(json.loads(line))
            if len(rows) >= limit:
                break
    return rows


VIETNAMESE_CHAR_PATTERN = re.compile(
    r"[ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]",
    re.IGNORECASE,
)
VIETNAMESE_HINT_WORDS = {
    "ở", "đi", "đến", "nên", "gì", "món", "ăn", "chơi", "lịch", "trình",
    "khách", "sạn", "địa", "điểm", "tham", "quan", "bao", "nhiêu", "ngày",
    "buổi", "sáng", "tối", "mùa", "nào", "đẹp", "tiện", "gợi", "ý",
}
ENGLISH_HINT_WORDS = {
    "what", "where", "when", "which", "how", "best", "travel", "trip", "food",
    "restaurant", "hotel", "stay", "itinerary", "attraction", "visit", "things",
    "to", "do", "in", "near", "around", "recommend", "guide",
}


def detect_query_language(query: str) -> str:
    """Detect nhanh ngôn ngữ query để tránh dịch lại query tiếng Anh."""

    text = query.strip().lower()
    if not text:
        return "unknown"
    if VIETNAMESE_CHAR_PATTERN.search(text):
        return "vi"
    tokens = re.findall(r"[a-zA-ZÀ-ỹ]+", text)
    if not tokens:
        return "unknown"
    vi_hits = sum(1 for token in tokens if token in VIETNAMESE_HINT_WORDS)
    en_hits = sum(1 for token in tokens if token in ENGLISH_HINT_WORDS)
    if vi_hits > en_hits:
        return "vi"
    if en_hits > 0:
        return "en"
    return "en"


def translate_query_to_english(question_text: str) -> dict[str, Any]:
    """Dịch query tiếng Việt sang query tiếng Anh ngắn gọn, giữ intent và địa danh."""

    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là bộ chuyển đổi truy vấn cho hệ thống retrieval du lịch Việt Nam. "
                "Hãy chuyển câu hỏi tiếng Việt thành một query tiếng Anh ngắn, giàu keyword, phù hợp để search trong knowledge base tiếng Anh. "
                "Giữ nguyên địa danh riêng của Việt Nam ở dạng phổ biến trong tiếng Anh nếu có. "
                "Chỉ trả về JSON hợp lệ."
            ),
        },
        {
            "role": "user",
            "content": (
                "Câu hỏi tiếng Việt:\n"
                f"{question_text}\n\n"
                "Trả về đúng schema JSON sau:\n"
                "{\n"
                '  "english_query": "...",\n'
                '  "detected_locations": ["..."],\n'
                '  "intent": "attraction|itinerary|cuisine|transportation|accommodation|weather|culture|nightlife|shopping|comparison|practical_tip|general"\n'
                "}"
            ),
        },
    ]
    raw = chat_client.chat(messages, temperature=0.0, max_tokens=300)
    parsed = extract_json_object(raw)
    parsed.setdefault("english_query", question_text)
    parsed.setdefault("detected_locations", [])
    parsed.setdefault("intent", "general")
    return parsed


def prepare_retrieval_query(question: str) -> dict[str, Any]:
    """Chuẩn bị query retrieval: tiếng Anh thì dùng thẳng, tiếng Việt thì dịch."""

    detected_language = detect_query_language(question)
    if detected_language == "en":
        return {
            "retrieval_query": question,
            "retrieval_query_language": "en",
            "translation_status": "skipped",
            "translation_metadata": {
                "english_query": question,
                "detected_locations": [],
                "intent": "general",
                "reason": "Input đã được detect là tiếng Anh nên bỏ qua bước dịch.",
            },
        }

    translation = translate_query_to_english(question)
    return {
        "retrieval_query": str(translation.get("english_query") or question),
        "retrieval_query_language": "mixed_by_query",
        "translation_status": "success",
        "translation_metadata": translation,
    }



def load_jsonl_file(path: Path) -> list[dict[str, Any]]:
    """??c file JSONL UTF-8."""

    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL l?i ? {path}:{line_number}: {exc}") from exc
    return rows


SEMANTIC_PARENTS = load_jsonl_file(PARENTS_PATH)
SEMANTIC_CHILDREN = load_jsonl_file(CHILDREN_PATH)
PARENT_BY_ID = {item["parent_id"]: item for item in SEMANTIC_PARENTS}
CHILD_BY_ID = {item["child_id"]: item for item in SEMANTIC_CHILDREN}

print("Semantic parents:", len(PARENT_BY_ID))
print("Semantic children:", len(CHILD_BY_ID))


def normalize_context_space(value: Any) -> str:
    """Chu?n h?a kho?ng tr?ng ?? so s?nh v? format context."""

    return re.sub(r"\s+", " ", str(value or "")).strip()


def enrich_hybrid_result(result: dict[str, Any]) -> dict[str, Any]:
    """B? sung child metadata v? parent summary cho m?t retrieval result."""

    enriched = dict(result)
    child = CHILD_BY_ID.get(str(result.get("chunk_id")))
    if not child:
        return enriched

    parent = PARENT_BY_ID.get(str(child.get("parent_id"))) or {}
    metadata = child.get("metadata") or {}

    enriched["child"] = child
    enriched["parent"] = parent
    enriched["parent_id"] = child.get("parent_id")
    enriched["heading"] = child.get("heading")
    enriched["heading_path"] = child.get("heading_path") or []
    enriched["child_type"] = child.get("child_type")
    enriched["source_spans"] = child.get("source_spans") or []
    enriched["source_text"] = child.get("source_text") or result.get("source_text")
    enriched["document_title"] = parent.get("clean_title") or parent.get("title") or result.get("document_title")
    enriched["source_url"] = metadata.get("source_url") or result.get("source_url")
    enriched["language"] = metadata.get("language") or result.get("language")
    enriched["parent_context_summary"] = parent.get("context_summary") or ""
    enriched["parent_summary_type"] = parent.get("summary_type")
    return enriched

def search_chunks(query_en: str) -> list[dict[str, Any]]:
    """Dùng query tiếng Anh để retrieve chunk từ baseline index."""

    if RETRIEVER_NAME == "hybrid":
        return retriever.search(query_en, top_k=TOP_K, candidate_k=CANDIDATE_K)
    return retriever.search(query_en, top_k=TOP_K)


def trim_text(text: str, max_chars: int) -> str:
    """Cắt text theo ký tự để giữ prompt gọn nhưng vẫn còn đủ ngữ cảnh."""

    clean_text = " ".join(str(text or "").split())
    if len(clean_text) <= max_chars:
        return clean_text
    return clean_text[:max_chars].rstrip() + "..."


def build_context(chunks: list[dict[str, Any]]) -> str:
    """Format retrieved chunks thành context cho GPT-4o-mini."""

    context_items: list[str] = []
    for item in chunks:
        content = trim_text(str(item.get("source_text") or ""), MAX_CHARS_PER_CHUNK)
        context_items.append(
            "\n".join(
                [
                    f"[Nguồn {item.get('rank')} | score={item.get('score')} | retriever={item.get('retriever')} ]",
                    f"Title: {item.get('document_title')}",
                    f"URL: {item.get('source_url')}",
                    f"Document ID: {item.get('document_id')}",
                    f"Chunk ID: {item.get('chunk_id')}",
                    f"Language: {item.get('language')}",
                    "Content:",
                    content,
                ]
            )
        )
    return "\n\n".join(context_items)


def answer_with_context(question_en: str, query_en: str, translation: dict[str, Any], chunks: list[dict[str, Any]]) -> str:
    """Sinh c?u tr? l?i ti?ng Vi?t t? c?u h?i test ti?ng Anh v? retrieved context."""

    context = build_context(chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "B?n l? AI Assistant du l?ch Vi?t Nam d?ng Retrieval-Augmented Generation. "
                "B?n tr? l?i b?ng ti?ng Vi?t c? d?u, t? nhi?n, r? r?ng v? h?u ?ch cho ng??i ?i du l?ch. "
                "Ch? s? d?ng th?ng tin trong CONTEXT ???c cung c?p. Kh?ng b?a th?ng tin, kh?ng t? th?m gi? v?, gi? m? c?a, "
                "l?ch s? ki?n, quy ??nh visa, th?i ti?t hi?n t?i ho?c th?ng tin d? thay ??i n?u CONTEXT kh?ng n?u r?. "
                "N?u CONTEXT kh?ng ?? ho?c kh?ng ??ng intent, h?y n?i r? d? li?u hi?n t?i ch?a ??. "
                "Cu?i c?u tr? l?i n?n c? m?c 'Ngu?n tham kh?o' n?u context c? URL."
            ),
        },
        {
            "role": "user",
            "content": (
                "CONTEXT:\n"
                f"{context}\n\n"
                "C?U H?I TEST TI?NG ANH:\n"
                f"{question_en}\n\n"
                "QUERY ?? D?NG ?? RETRIEVE:\n"
                f"{query_en}\n\n"
                "TH?NG TIN PH?N T?CH QUERY:\n"
                f"{json.dumps(translation, ensure_ascii=False)}\n\n"
                "Y?U C?U:\n"
                "- Tr? l?i tr?c ti?p nhu c?u trong c?u h?i test ti?ng Anh, nh?ng di?n ??t b?ng ti?ng Vi?t c? d?u.\n"
                "- ?u ti?n g?i ? th?c t? cho chuy?n ?i.\n"
                "- N?u context ch? li?n quan m?t ph?n, h?y n?u r? ph?n n?o ch?c ch?n v? ph?n n?o thi?u d? li?u.\n"
                "- Kh?ng d?ng ki?n th?c ngo?i context.\n"
                "- Li?t k? t?i ?a 3 ngu?n tham kh?o ? cu?i n?u c? URL."
            ),
        },
    ]
    return chat_client.chat(messages, temperature=0.2, max_tokens=1000).strip()

def compact_sources(chunks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Lưu metadata nguồn gọn nhẹ trong output JSON."""

    sources: list[dict[str, Any]] = []
    for item in chunks:
        sources.append(
            {
                "rank": item.get("rank"),
                "score": item.get("score"),
                "retriever": item.get("retriever"),
                "chunk_id": item.get("chunk_id"),
                "document_id": item.get("document_id"),
                "document_title": item.get("document_title"),
                "source_url": item.get("source_url"),
                "language": item.get("language"),
                "text_preview": trim_text(str(item.get("source_text") or ""), 350),
            }
        )
    return sources

Semantic parents: 281
Semantic children: 1844


In [47]:
# =========================
# 5. Chạy pipeline và xuất JSON
# =========================

queries = load_queries(INPUT_QUERY_PATH, LIMIT)
results: list[dict[str, Any]] = []

for index, row in enumerate(queries, start=1):
    query_id = row.get("query_id") or f"query_{index:04d}"
    question_en = row.get("query_en") or row.get("query") or row.get("question") or ""
    print(f"[{index}/{len(queries)}] {query_id}: {question_en}")

    step_errors: list[str] = []
    translation = {}
    retrieval_query = question_en
    retrieval_query_language = detect_query_language(question_en)
    translation_status = "not_started"
    chunks = []
    answer_en = ""

    try:
        query_payload = prepare_retrieval_query(question_en)
        retrieval_query = str(query_payload["retrieval_query"])
        retrieval_query_language = str(query_payload["retrieval_query_language"])
        translation_status = str(query_payload["translation_status"])
        translation = dict(query_payload["translation_metadata"])
    except Exception as exc:
        step_errors.append(f"translation_error: {exc}")
        translation = {
            "english_query": question_en,
            "detected_locations": row.get("locations") or [],
            "intent": row.get("user_intent") or row.get("category") or "general",
            "fallback_reason": "Không dịch được query bằng LLM, dùng query gốc để retrieval.",
        }
        retrieval_query = question_en
        retrieval_query_language = detect_query_language(question_en)
        translation_status = "failed"

    try:
        chunks = search_chunks(retrieval_query)
    except Exception as exc:
        step_errors.append(f"retrieval_error: {exc}")

    if chunks:
        try:
            answer_en = answer_with_context(question_en, retrieval_query, translation, chunks)
        except Exception as exc:
            step_errors.append(f"answer_error: {exc}")
    else:
        step_errors.append("answer_skipped: Không có retrieved chunks để sinh câu trả lời.")

    status = "success" if answer_en and not step_errors else "partial" if chunks else "error"
    error = " | ".join(step_errors) if step_errors else None

    results.append(
        {
            "query_id": query_id,
            "status": status,
            "error": error,
            "question_en": question_en,
            "query_en": retrieval_query if retrieval_query_language == "en" else "",
            "retrieval_query": retrieval_query,
            "retrieval_query_language": retrieval_query_language,
            "translation_status": translation_status,
            "translation_metadata": translation,
            "answer_en": answer_en,
            "sources": compact_sources(chunks),
            "input_metadata": {
                "user_intent": row.get("user_intent"),
                "category": row.get("category"),
                "locations": row.get("locations"),
                "traveler_profile": row.get("traveler_profile"),
                "time_context": row.get("time_context"),
            },
        }
    )

    time.sleep(REQUEST_DELAY_SECONDS)

payload = {
    "metadata": {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "input_query_path": str(INPUT_QUERY_PATH),
        "output_path": str(OUTPUT_PATH),
        "limit": LIMIT,
        "retriever": RETRIEVER_NAME,
        "top_k": TOP_K,
        "candidate_k": CANDIDATE_K,
        "chat_model": CHAT_MODEL,
        "api_base_url": BASE_URL,
        "retrieval_query_language": "en",
        "chunking_strategy": "semantic_parent_child",
        "retrieval_unit": "child",
        "parent_embedding": False,
        "parent_in_faiss": False,
        "semantic_index_dir": str(SEMANTIC_INDEX_DIR),
        "answer_language": "vi",
        "success_count": sum(1 for item in results if item["status"] == "success"),
        "partial_count": sum(1 for item in results if item["status"] == "partial"),
        "error_count": sum(1 for item in results if item["status"] == "error"),
    },
    "results": results,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Đã lưu output:", OUTPUT_PATH)
print(
    "Success:", payload["metadata"]["success_count"],
    "| Partial:", payload["metadata"]["partial_count"],
    "| Error:", payload["metadata"]["error_count"],
)
payload["results"][:1]

[1/100] traveler_need_q_0001: Where should I stay in Hue for 2 days and 1 night for a business trip combined with leisure travel? Recommend convenient areas for sightseeing, food, and accommodation options. Please make it practical and not too generic.
[2/100] traveler_need_q_0002: What are the best attractions or places to visit in Phong Nha for 2 days and 1 night for a group of friends, especially around temples and pagodas. Please make it easy for first-time visitors.
[3/100] traveler_need_q_0003: Can you suggest a practical itinerary for Hoi An in the morning for someone who likes local experiences, with priority activities and places to visit?
[4/100] traveler_need_q_0004: What should I eat in Da Lat during the rainy season for a couple? Recommend local dishes, food experiences, and places related to street food. Please make it include things to avoid or practical cautions.
[5/100] traveler_need_q_0005: Where can I experience local culture, heritage, or history in Ninh Binh for a 

[{'query_id': 'traveler_need_q_0001',
  'status': 'success',
  'error': None,
  'question_en': 'Where should I stay in Hue for 2 days and 1 night for a business trip combined with leisure travel? Recommend convenient areas for sightseeing, food, and accommodation options. Please make it practical and not too generic.',
  'query_en': 'Where should I stay in Hue for 2 days and 1 night for a business trip combined with leisure travel? Recommend convenient areas for sightseeing, food, and accommodation options. Please make it practical and not too generic.',
  'retrieval_query': 'Where should I stay in Hue for 2 days and 1 night for a business trip combined with leisure travel? Recommend convenient areas for sightseeing, food, and accommodation options. Please make it practical and not too generic.',
  'retrieval_query_language': 'en',
  'translation_status': 'skipped',
  'translation_metadata': {'english_query': 'Where should I stay in Hue for 2 days and 1 night for a business trip combin